# Test de NMF pour l'analyse de topic

## Chargement des librairies nécessaires

In [9]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF


In [21]:
df=pd.read_csv("corpus_zola_nettoye.csv") # Chargement du DataFrame nettoyé à partir du fichier CSV

In [22]:
def segmenter_texte(text, taille=500): # On segmente le texte en segments de 300 mots
    mots = text.split() # On divise le texte en mots
    segments = [] # Liste pour stocker les segments de texte
    for i in range(0, len(mots), taille): # On boucle sur les mots par pas de "taille" (300 mots)
        segment = " ".join(mots[i:i+taille]) # On crée un segment en joignant les mots du segment
        if len(segment.split()) >= 100:  # On vérifie que le segment contient au moins 100 mots pour éviter les segments trop courts
            segments.append(segment) # On ajoute le segment à la liste des segments
    return segments # On retourne la liste des segments de texte

lignes = [] # Liste pour stocker les segments de texte avec leurs titres et IDs

for _, row in df.iterrows():
    titre = row["nom_fichier"] # On utilise le nom du fichier comme titre
    texte = row["texte_nettoye"] # On segmente le texte en segments de 300 mots
    segments = segmenter_texte(texte) # On ajoute chaque segment à la liste avec son titre et son ID
    
    for j, seg in enumerate(segments):
        
        
        lignes.append({ 
            "titre": titre, # Titre du texte (nom du fichier)
            "segment_id": j, # ID du segment (index du segment dans le texte) 
            "texte_segment": seg # Contenu du segment de texte
        })

df_segments = pd.DataFrame(lignes) # Création d'un DataFrame à partir de la liste de segments

df_segments

,titre,segment_id,texte_segment
0,1893_20_Le_docteur_Pascal._clean.txt,0,chaleur ardent après-midi juillet salle volet ...
1,1893_20_Le_docteur_Pascal._clean.txt,1,œuvre beau moment unique servant vrai maîtress...
2,1893_20_Le_docteur_Pascal._clean.txt,2,quartier campagne lieu ville cure éclatant hon...
3,1893_20_Le_docteur_Pascal._clean.txt,3,gloire conversion haut lutte besogne saint exa...
4,1893_20_Le_docteur_Pascal._clean.txt,4,rose orange rideau fenêtre lit exquis pourpre ...
...,...,...,...
2159,1894_1_Lourdes._clean.txt,88,sommeil hasard bras membre joindre lèvre tiède...
2160,1894_1_Lourdes._clean.txt,89,horreur spectacle ennui simplicité triste vie ...
2161,1894_1_Lourdes._clean.txt,90,rose songe plein paradis légende jeunesse libr...
2162,1894_1_Lourdes._clean.txt,91,voyant sœur sœur charité corps exposé jour fou...


In [23]:
df_segments["nb_tokens_nettoyes"] = df_segments["texte_segment"].apply(lambda x: len(x.split())) # Calcul du nombre de tokens nettoyés pour chaque segment
df_segments["nb_tokens_nettoyes"].describe() # Affichage des statistiques descriptives du nombre de tokens nettoyés par segment (moyenne, écart-type, min, max, etc.)

count    2164.000000
mean      497.950092
std        22.618983
min       145.000000
25%       500.000000
50%       500.000000
75%       500.000000
max       500.000000
Name: nb_tokens_nettoyes, dtype: float64

In [24]:
df_segments

,titre,segment_id,texte_segment,nb_tokens_nettoyes
0,1893_20_Le_docteur_Pascal._clean.txt,0,chaleur ardent après-midi juillet salle volet ...,500
1,1893_20_Le_docteur_Pascal._clean.txt,1,œuvre beau moment unique servant vrai maîtress...,500
2,1893_20_Le_docteur_Pascal._clean.txt,2,quartier campagne lieu ville cure éclatant hon...,500
3,1893_20_Le_docteur_Pascal._clean.txt,3,gloire conversion haut lutte besogne saint exa...,500
4,1893_20_Le_docteur_Pascal._clean.txt,4,rose orange rideau fenêtre lit exquis pourpre ...,500
...,...,...,...,...
2159,1894_1_Lourdes._clean.txt,88,sommeil hasard bras membre joindre lèvre tiède...,500
2160,1894_1_Lourdes._clean.txt,89,horreur spectacle ennui simplicité triste vie ...,500
2161,1894_1_Lourdes._clean.txt,90,rose songe plein paradis légende jeunesse libr...,500
2162,1894_1_Lourdes._clean.txt,91,voyant sœur sœur charité corps exposé jour fou...,500


In [51]:
vectorizer = TfidfVectorizer( 
    min_df=5,
    max_df=0.8
    )
# min_df=5 : Ignorer les termes qui apparaissent dans moins de 3 segments, car ils sont considérés comme peu informatifs.
# max_df=0.8 : Ignorer les termes qui apparaissent dans plus de 80% des segments, car ils sont trop fréquents et ne contribuent pas à différencier les segments.

X = vectorizer.fit_transform(df_segments["texte_segment"])  
#Appliquer le TF-IDF vectorizer sur les segments de texte nettoyés pour obtenir une matrice de caractéristiques (termes pondérés par leur importance dans les segments).

X.shape

(2164, 9276)

In [50]:
def afficher_topics(model, feature_names, n_top_words=15):
    for topic_idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]]
        print(f"Topic {topic_idx+1} : {' | '.join(top_words)}")



feature_names = vectorizer.get_feature_names_out()

#for k in  [8, 10, 12, 15, 18 ]: # On teste différents nombres de topics (de 5 à 15) pour trouver le nombre optimal de topics pour notre analyse.
    
    #print(f"\n NMF avec {k} topics") 
nmf = NMF(n_components=12, 
          random_state=42,
          init="nndsvda",
          solver="cd",
          max_iter=500,
          
            )
W = nmf.fit_transform(X)
afficher_topics(nmf, feature_names)

print("Erreur :", nmf.reconstruction_err_)




Topic 1 : monsieur | dame | salon | comte | porte | chambre | comtesse | deberl | empereur | voix | jeune | mère | fille | table | docteur
Topic 2 : enfant | travail | vie | usine | mère | fils | fille | père | œuvre | bonheur | heureux | année | terre | maison | famille
Topic 3 : prussien | soldat | armée | général | route | cheval | empereur | canon | capitaine | troupe | corps | ville | obu | feu | régiment
Topic 4 : soleil | arbre | blanc | herbe | rose | ciel | eau | fleur | jardin | ombre | haut | feuille | bleu | milieu | branche
Topic 5 : abbé | prêtre | curé | église | trouche | jardin | sous | monsieur | autel | soutane | docteur | préfecture | frère | maison | évêque
Topic 6 : franc | argent | affaire | bourse | rue | million | vendeur | magasin | fortune | maison | jeune | somme | chiffre | client | billet
Topic 7 : coupeau | rue | boutique | maman | vin | verre | table | quartier | zingueur | vieux | père | blanchisseur | gros | lit | marchand
Topic 8 : jeune | chambre | a